# Mamba (S6) — a toy-scale build of a Selective State Space Model

A minimal, from-scratch implementation of a **Mamba** block (Gu & Dao,
*"Mamba: Linear-Time Sequence Modeling with Selective State Spaces,"* 2023),
small enough to train end-to-end on a free Colab GPU (or CPU) in a couple of
minutes.

Companion write-up: see `README.md` in this same folder for the concepts
before the code.

## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## 1. The big picture

Mamba is built around a **State Space Model (SSM)** — a very old idea from
control theory, repurposed as a sequence layer. The core is a hidden state
`h` that gets updated at every timestep and read out to produce the output:

```
h_t = A_bar * h_{t-1} + B_bar * x_t
y_t = C_t * h_t + D * x_t
```

This looks a lot like an RNN, and it is one — but the "S" in S6 stands for
**selective**: unlike a classic SSM (whose `A`, `B`, `C` matrices are fixed
after training), Mamba makes `B`, `C`, and the step size `Δ` *functions of
the current input*. That's what lets the model decide, per token, whether to
let information into the state, keep it, or ignore it — the SSM equivalent
of an attention mechanism's "relevance."

A full Mamba block wraps this SSM recurrence with:

- an **input projection** that splits into two branches: one goes through
  the SSM, the other is a plain gate,
- a **short causal convolution** before the SSM, so nearby tokens can mix
  locally first (same idea as KDA's short conv),
- a **SiLU gate** from the second branch, multiplied into the SSM's output
  before the final projection back down.

## 2. What "selective" actually changes, step by step

At every timestep `t`, three input-dependent values are computed from the
token:

- `Δ_t` (delta) — a positive step size (via `softplus`), one value per
  inner channel. Think of it as "how much time has passed" for this token —
  a large `Δ_t` means the state changes a lot, a small one means the token
  barely moves the state.
- `B_t` — which parts of the incoming token get written into the state.
- `C_t` — which parts of the state get read out as the output.

These get combined with a *fixed, learned* per-channel matrix `A` (always
negative, so the state naturally decays) to produce **discretized**,
per-timestep versions of the recurrence:

```
A_bar_t = exp(Δ_t * A)              # decay factor for this specific token
B_bar_t = Δ_t * B_t                 # how much of x_t actually gets written in
h_t = A_bar_t * h_{t-1} + B_bar_t * x_t
y_t = C_t · h_t + D * x_t
```

Every one of `Δ_t`, `B_t`, `C_t` depends on the token itself — that's the
entire "selection mechanism." A classic (non-selective) SSM would use the
same `A_bar`, `B_bar` at every timestep regardless of input; Mamba recomputes
them fresh for every token.

> **Simplification used here:** the real implementation uses a **parallel
> scan** algorithm to compute this recurrence efficiently on a GPU, plus a
> custom CUDA kernel that fuses the discretization and recurrence into one
> pass. This notebook uses the plain **sequential recurrence** — a Python
> `for` loop over timesteps — which computes the exact same math, just much
> slower and far easier to read.

In [ ]:
class ShortConv(nn.Module):
    def __init__(self, dim, kernel_size=4):
        super().__init__()
        self.kernel_size = kernel_size
        self.conv = nn.Conv1d(dim, dim, kernel_size, groups=dim, padding=0)
    def forward(self, x):  # x: [B,T,D]
        x = x.transpose(1, 2)
        x = F.pad(x, (self.kernel_size - 1, 0))   # left-pad only -> causal
        return self.conv(x).transpose(1, 2)

In [ ]:
class MambaBlock(nn.Module):
    def __init__(self, d_model=64, d_inner=128, d_state=16, dt_rank=8, conv_kernel=4):
        super().__init__()
        self.d_inner, self.d_state = d_inner, d_state
        self.in_proj = nn.Linear(d_model, 2 * d_inner, bias=False)   # splits into x-branch, gate-branch
        self.conv = ShortConv(d_inner, conv_kernel)
        self.x_proj = nn.Linear(d_inner, dt_rank + 2 * d_state, bias=False)   # predicts Δ, B, C
        self.dt_proj = nn.Linear(dt_rank, d_inner, bias=True)
        # A is fixed after training but still learned: init as -1,-2,...,-d_state per channel (S4D-style)
        self.A_log = nn.Parameter(torch.log(torch.arange(1, d_state + 1, dtype=torch.float32))
                                   .unsqueeze(0).repeat(d_inner, 1))
        self.D = nn.Parameter(torch.ones(d_inner))          # skip connection (Eq. y_t = ... + D*x_t)
        self.out_proj = nn.Linear(d_inner, d_model, bias=False)
        self.dt_rank = dt_rank

    def forward(self, x):
        B, T, Dm = x.shape
        xz = self.in_proj(x)
        x_b, z_b = xz.chunk(2, dim=-1)                       # SSM branch, gate branch
        x_b = F.silu(self.conv(x_b))

        dbc = self.x_proj(x_b)
        dt, Bm, Cm = torch.split(dbc, [self.dt_rank, self.d_state, self.d_state], dim=-1)
        delta = F.softplus(self.dt_proj(dt))                 # B,T,d_inner -- the selective step size
        A = -torch.exp(self.A_log)                           # d_inner,d_state -- always negative (stable decay)

        h = x.new_zeros(B, self.d_inner, self.d_state)
        ys = []
        for t in range(T):                                    # sequential recurrence, one token at a time
            dt_t = delta[:, t]
            Bt, Ct, xt = Bm[:, t], Cm[:, t], x_b[:, t]
            A_bar = torch.exp(dt_t.unsqueeze(-1) * A.unsqueeze(0))              # discretized decay
            dBx = dt_t.unsqueeze(-1) * Bt.unsqueeze(1) * xt.unsqueeze(-1)       # discretized input write
            h = A_bar * h + dBx
            y_t = (h * Ct.unsqueeze(1)).sum(-1) + self.D * xt
            ys.append(y_t)
        y = torch.stack(ys, dim=1)
        y = y * F.silu(z_b)                                    # gate branch
        return self.out_proj(y)

## 3. Assembling a tiny language model

A minimal 2-layer causal LM: embed tokens, run each through a `MambaBlock` +
a small feed-forward layer (both with pre-norm residual connections), then
project to vocabulary logits.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        norm = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(norm + self.eps) * self.weight

class SwiGLU(nn.Module):
    def __init__(self, d, hidden_mult=2):
        super().__init__()
        h = d * hidden_mult
        self.Wg = nn.Linear(d, h, bias=False)
        self.Wu = nn.Linear(d, h, bias=False)
        self.Wd = nn.Linear(h, d, bias=False)
    def forward(self, x):
        return self.Wd(F.silu(self.Wg(x)) * self.Wu(x))

class TinyLM(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([MambaBlock(d_model) for _ in range(n_layers)])
        self.mlps = nn.ModuleList([SwiGLU(d_model) for _ in range(n_layers)])
        self.norms1 = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.norms2 = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx):
        x = self.embed(idx)
        for blk, mlp, n1, n2 in zip(self.blocks, self.mlps, self.norms1, self.norms2):
            x = x + blk(n1(x))
            x = x + mlp(n2(x))
        return self.lm_head(self.final_norm(x))

## Proving it actually works

Everything above is only worth something if gradients actually flow correctly
through Mamba once it's wired into a real model. So the rest of this
notebook:

1. wraps Mamba into a tiny 2-layer causal language model,
2. builds a **tiny synthetic dataset** (a repeating `"0123456789ABCDEF"`
   string — enough to check the model can learn *any* sequential structure
   at all, no real corpus needed),
3. runs **one forward + backward pass** as a sanity check (right output
   shape, no `NaN` gradients),
4. **trains for a few hundred steps**, and
5. **generates** from the trained model — if training worked, the output
   should show visible periodicity.

This is deliberately not a "real" training run. It exists purely to catch
architecture bugs, which is the whole point of a toy-scale build.

In [ ]:
# --- synthetic dataset ---
pattern = "0123456789ABCDEF"      # synthetic, no copyright concerns
text = pattern * 200
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
vocab_size = len(chars)
max_seq_len = 32

model = TinyLM(vocab_size).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model built. Trainable parameters: {n_params:,}")

In [ ]:
# --- sanity check: one forward + backward pass before training ---
xb0 = data[:max_seq_len].unsqueeze(0).to(device)
yb0 = data[1:max_seq_len + 1].unsqueeze(0).to(device)
out0 = model(xb0)
logits0 = out0[0] if isinstance(out0, tuple) else out0
print(f"Sanity check -- logits shape: {tuple(logits0.shape)} (expect [1, {max_seq_len}, {vocab_size}])")
loss0 = F.cross_entropy(logits0.reshape(-1, vocab_size), yb0.reshape(-1))
if isinstance(out0, tuple):
    loss0 = loss0 + out0[1]
loss0.backward()
n_nan_grads = sum(torch.isnan(p.grad).any().item() for p in model.parameters() if p.grad is not None)
print(f"Sanity check -- initial loss: {loss0.item():.4f}, NaN grads: {n_nan_grads}")
model.zero_grad()

In [ ]:
# --- training loop ---
def get_batch(data, block_size, batch_size, device):
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
n_steps, batch_size = 300, 16
print("Training on synthetic periodic sequence (verifies grads flow end-to-end)...")
for step in range(n_steps):
    xb, yb = get_batch(data, max_seq_len, batch_size, device)
    out = model(xb)
    logits = out[0] if isinstance(out, tuple) else out
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1))
    if isinstance(out, tuple):
        loss = loss + out[1]
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    if step % 50 == 0 or step == n_steps - 1:
        print(f"  step {step:4d} | loss {loss.item():.4f}")

In [ ]:
# --- generation ---
@torch.no_grad()
def generate(model, start_idx, n_new):
    model.eval()
    idx = start_idx.clone()
    for _ in range(n_new):
        out = model(idx)
        logits = out[0] if isinstance(out, tuple) else out
        probs = F.softmax(logits[:, -1, :], dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    model.train()
    return idx

start = data[:8].unsqueeze(0).to(device)
gen = generate(model, start, 48)[0].tolist()
print("Generated (should show visible periodicity if training worked):")
print(''.join(itos[i] for i in gen))

## Where to go from here

- **Replace the sequential loop with a parallel scan** — the recurrence
  `h_t = A_bar_t * h_{t-1} + B_bar_t * x_t` is an associative scan, which
  means it can be computed in `O(log T)` parallel steps instead of `O(T)`
  sequential ones. This is the change that would make it fast.
- **Try a non-diagonal `A`** — this notebook uses a diagonal state matrix
  (S4D-style) for simplicity; the original S4 paper uses a structured
  (HiPPO) matrix that's better at capturing long-range dependencies.
- **Stack more layers and compare against the KDA notebook** on the same
  toy task — both are linear-time recurrent alternatives to attention, but
  arrive at the "selective forget gate" idea differently.

Reference: Gu & Dao, *"Mamba: Linear-Time Sequence Modeling with Selective
State Spaces,"* 2023.